In [160]:
from langchain_groq import ChatGroq
from langchain_core.runnables import RunnableLambda

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser,CommaSeparatedListOutputParser

from langchain_core.prompts.few_shot import FewShotChatMessagePromptTemplate


In [162]:

#     {
#         "input":"",
#         "output":""
#     }
#     ,
#     {
#         "input":"",
#         "output":""
#     },
#     {
#         "input":"",
#         "output":""
#     },
#     {
#         "input":"",
#         "output":""
#     },
#     {
#         "input":"",
#         "output":""
#     },
#     {
#         "input":"",
#         "output":""
#     },

In [163]:
# Few Shot Examples
examples = [
    {
        "input": "Monday May 28 - Morning Report: National Guardsman missing after monstrous floods sink Maryland town; Trump visits Arlington after critics slam appalling Memorial Day tweet Via @nypost",
        "output": "Arlington,Maryland"
    },
    {
        "input": "Sgt. Eddison Hermond is missing after being swept away by historic flooding in Ellicott City, Maryland. Witnesses say he was trying to help a woman rescue her cat",
        "output": "Ellicott City,Maryland"
    },
    {
        "input":"Heart Breaking views of damaged roads at Jaltan Road, Mirpur Kashmir May Allah protect us all #Ameen I request all the people, please do something practical for the relief operations in the affected areas #Earthquake",
        "output":"Jaltan Road, Mirpur,Kashmir"
    },
 
    {
        "input":"@WhiteHouse @realDonaldTrump @GovKemp @GovRonDeSantis We still have storm damage from Florence while our POA is illegally withholding insurance funds! We have $36000 in damages and two toxic molds! Dorian is coming up the coastline now and here we are! #northcarolina #oakislandnc #veteran #veteranfamily",
        "output":"Florence,northcarolina"
    },
    {
        "input":"The magnitude of the devastation caused by the #earthquake in Mirpur, Azad Kashmir, is frightening. Over 21 people are reported dead so far. More than 400 injured.",
        "output":"Mirpur,Azad Kashmir"
    },
    {
        "input":"Top story: Icing on the Cake Los Gatos on Instagram: To help aid our California neighbors impacted by fires, we’re selling raffle tickets through November 30, with all proceeds going directly” , see more",
        "output":"California"
    },
    {
        "input":"Flash Flood Warning until until 6:15 AM for Brazoria, Fort Bend, Galveston & Harris counties. #Harvey is still here. Stay Safe! #KHOU11",
        "output":"Brazoria,Fort Bend, Galveston,Harris"
    },
    {
        "input":"San Diego-Based USS Lake Erie Assists Flood Victims in Sri Lanka  #news",
        "output":"San Diego,Sri Lanka"
    },
    {
        "input":"RT @ItsYakin: Lulu Group in #UAE donated US$ 100,000 for #FloodSL relief operations. The company is planning to enter our #retail market in",
        "output":"UAE"
    }
    ,
    {
        "input":"RT @SteveBowenWx: Latest numbers for Sri Lanka flood event: - 292 dead/missing - 150 injured - 21,205 homes damaged/destroyed - $197M econo",
        "output":"Sri Lanka"
    }


]

In [164]:
Instructions = """
You are an expert in custom named entity recognition and tweet analysis. Given the following instructions, extract the entities for the specified labels from the provided text and present the results in a list format.

### Instructions:

1. **Extract Entities Exactly as Mentioned**: Entities must be extracted exactly as they appear in the text.
2. **Return Each Entity Under Its Label**: Organize the entities under their respective labels without creating new labels.
3. **Provide a List of Entities for Each Label**: Ensure that if no entities are found for a label, an empty list is returned.
4. **Maintain Order**: Return the entities in the order they appear in the sentence.
5. **Exclude Organizations**: Do not include names of organizations.
6. **Ignore Hashtags and Places**: Hashtag names or places are not to be considered.
7. **Avoid Abbreviations**: Do not include abbreviations at the end of tweets, as those are tags.
8. **Focus on Accuracy and Relevance**: Accuracy and relevance in your responses are key.
9. **Return Only What is Asked**: Do not assume anything; return only the entities requested.

### Possible Entities:

- District
- Neighborhood
- Natural Point-of-Interest
- Road/street
- County
- Continent
- Other locations
- Country
- Human-made Point-of-Interest
- City/town
- State
- Island

---

Please follow these instructions carefully to ensure the entities are extracted and presented correctly.
"""


In [165]:
# Instructions = """

# You are an expert in custom named entity recognition and tweet analysis. 
# Given the following instructions.
# Extract the entities for the following labels from the given text and provide the results in a list like structure
# - Entities must be extracted exactly as mentioned in the text.
# - Return each entity under its label without creating new labels.
# - Provide a list of entities for each label, ensuring that if no entities are found for a label, an empty list is returned.
# -Return the entity in order they are in the sentence
# - Don't include names of organisations.
# - Hashtag names or places are not to be considered
# - Accuracy and relevance in your responses are key.
# - Don't assume anything return only what is asked
# - Don't return abreviation at the end of tweets, those are tags

# Here are the possible entities: 
# District", "Neighborhood", "Natural Point-of-Interest", "Road/street", "County", "Continent", "Other locations", "Country", "Human-made Point-of-Interest", "City/town", "State", "Island"


# """

# District", "Neighborhood", "Natural Point-of-Interest", "Road/street", "County", "Continent", "Other locations", "Country", "Human-made Point-of-Interest", "City/town", "State", "Island"
# Labels and their Descriptions:
# - District: A subdivision of a larger administrative unit, such as a county or province. Extract information related to a name of a District.
# - Neighborhood: A local area within a city or town. Extract information related to a name of a Neighborhood.
# - Natural Point-of-Interest: A naturally occurring feature or landmark, such as a mountain or lake. Extract information related to a name of a Natural Point-of-Interest.
# - Road/street: A public thoroughfare for travel by vehicles. Extract information related to a name of a Road or Street.
# - County: A subdivision of a state or province. Extract information related to a name of a County.
# - Continent: One of the seven continents on Earth. Extract information related to a name of a Continent.
# - Other locations: Any location that does not fit into one of the other categories. Extract information related to a name of a location that does not fit into one of the other categories.
# - Country: A sovereign state or nation. Extract information related to a name of a Country.
# - Human-made Point-of-Interest: A man-made feature or landmark, such as a building or monument. Extract information related to a name of a Human-made Point-of-Interest.
# - City/town: A municipality or urban area. Extract information related to a name of a City or Town.
# - State: A subdivision of a country. Extract information related to a name of a State.
# - Island: A piece of land surrounded by water. Extract information related to a name of an Island.

# Here are some few examples you can refer on before you return the list:


In [166]:
# We now transform these to example messages
example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)

# Few shot examples
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

# prompt creation
prompt = ChatPromptTemplate.from_messages([
    ("system", Instructions),
    few_shot_prompt,
    ("user", "{question}"),
])



# STEP BACK FUNCTION
def step_back_prompt(question):

    chain = prompt | llm | CommaSeparatedListOutputParser()

    answer = chain.invoke({"question": question})
    

    return answer

response=step_back_prompt("When did Bwenghe kyangwi Graduated?")

In [168]:
question="Rashid Hussains mother is appealing @UN and other humanitarian Organizations to save her son and save his life. #RashidHussain was abducted by the #UAE #Pakistan after missing more than sex months. @Advjalila @QadeerMama #SaveRashidHussain"
feedback = step_back_prompt(question)

print(feedback)

['UAE', 'Pakistan']


In [169]:
import pandas as pd
test = pd.read_csv('Test.csv')

test.head(10)


,tweet_id,text
0,ID_1001154804658286592,What is happening to the infrastructure in New...
1,ID_1001155505459486720,SOLDER MISSING IN FLOOD.. PRAY FOR EDDISON HER...
2,ID_1001155756371136512,RT @TIME: Police searching for missing person ...
3,ID_1001159445194399744,Flash Flood Tears Through Maryland Town For Se...
4,ID_1001164907587538944,Ellicott City #FLOODING Pictures: Maryland Gov...
5,ID_1001178904617476096,@CBSNews Our Harts gos out to a Fellow Soldier...
6,ID_1001179909245587456,CRAZY VIDEO. Roaring flash #floods struck a #M...
7,ID_1001180876548591616,I liked a @YouTube video BREAKING: Devastatin...
8,ID_1001182906130280448,Thank you to the first responders who are taki...
9,ID_1001185240256311296,CBS News: Ellicott City floods: Maryland offic...


In [170]:
import pandas as pd
from tqdm import tqdm

# Initialize tqdm to work with pandas
tqdm.pandas()

# Assuming the DataFrame test is already defined and step_back_prompt is your function
test['Locations'] = test['text'].progress_apply(lambda question: ' '.join(step_back_prompt(question)))


100%|██████████| 2942/2942 [3:10:49<00:00,  3.89s/it]    


In [172]:
test.Locations.head(10)

0      New England Maryland
1    Ellicott City Maryland
2    Ellicott City Maryland
3    Ellicott City Maryland
4    Ellicott City Maryland
5                  Maryland
6                  Maryland
7    Ellicott City Maryland
8    Ellicott City Maryland
9    Ellicott City Maryland
Name: Locations, dtype: object

In [173]:
test.to_csv('output1.csv',index=False)

In [174]:
test.columns

Index(['tweet_id', 'text', 'Locations'], dtype='object')

In [176]:
output_1=test[['tweet_id','Locations']]

In [181]:
# output_1.rename(columns={'tweet_id':'ID'})

output_1.to_csv('output2.csv',index=False)

In [178]:
has_nulls = test['Locations'].isnull().any()

print(f"Are there null values in the 'Locations' column? {has_nulls}")

Are there null values in the 'Locations' column? False
